In [1]:
import pandas as pd
from sklearn.decomposition import PCA

# 1. 데이터프레임 생성
df = pd.DataFrame({
    '키': [160, 165, 172, 168, 180],
    '몸무게': [55, 63, 60, 75, 68],
    '나이': [20, 35, 25, 45, 30]
})

display(df)

# 2. PCA: 변수 3개를 2개로 축소
pca = PCA(n_components=2)
X_pca = pca.fit_transform(df)

# 3. 결과 확인
df_pca = pd.DataFrame(X_pca, columns=['주성분1', '주성분2'])
display(df_pca)

# 4. 각 주성분의 설명력
display(pca.explained_variance_ratio_) 
# [0.72644472, 0.26400856] 첫번째 변수가 73%, 두번째 변수가 26%, 합하면 두변수만으로도 99%정도 설명


,키,몸무게,나이
0,160,55,20
1,165,63,35
2,172,60,25
3,168,75,45
4,180,68,30


,주성분1,주성분2
0,-16.146057,-4.909677
1,1.241246,-5.265560
2,-6.308937,4.708793
3,16.849193,-5.357561
4,4.364555,10.824005


array([0.72644472, 0.26400856])

In [2]:
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score

# ==================================================
# 1. 데이터 로드
# ==================================================

# 유방암 데이터셋 로드
# X: 30개 특성, y: 양성 또는 악성 클래스
data = load_breast_cancer()

X = pd.DataFrame(data.data, columns=data.feature_names)
y = data.target

# 데이터의 크기와 앞부분 확인
print('전체 데이터 크기:', X.shape)
print('전체 특성 개수:', X.shape[1])
display(X.head())

전체 데이터 크기: (569, 30)
전체 특성 개수: 30


,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst radius,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension
0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,0.07871,...,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,...,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,0.05999,...,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758
3,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,0.09744,...,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300
4,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,0.05883,...,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678


In [3]:
# ==================================================
# 2. 학습 데이터와 테스트 데이터 분리
# ==================================================

# 학습 데이터 80%, 테스트 데이터 20%
# 두 모델에 동일한 데이터를 사용하여 성능을 비교
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


In [5]:
# ==================================================
# 3. 전체 30개 특성으로 분류 모델 학습
# ==================================================

# RandomForestClassifier 모델 생성
model1 = RandomForestClassifier(
    n_estimators=100,  # 의사결정나무 100개 사용
    random_state=42    # 실행 결과 재현
)

# 30개 특성을 이용하여 학습
model1.fit(X_train, y_train)

# 테스트 데이터의 클래스 예측
pred1 = model1.predict(X_test)

In [6]:
# ==================================================
# 4. PCA 적용: 30개 특성 → 2개 주성분
# ==================================================

# PCA는 변수의 스케일에 영향을 받으므로 표준화
scaler = StandardScaler()

# 학습 데이터에서 평균과 표준편차를 계산하고 표준화
X_train_scaled = scaler.fit_transform(X_train)

# 테스트 데이터는 학습 데이터의 평균과 표준편차로 표준화
X_test_scaled = scaler.transform(X_test)

# PCA 모델 생성: 주성분 2개 추출
pca = PCA(n_components=2)

# 학습 데이터로 PCA를 학습하고 2개 주성분으로 변환
X_train_pca = pca.fit_transform(X_train_scaled)

# 테스트 데이터도 학습된 PCA 기준으로 변환
X_test_pca = pca.transform(X_test_scaled)

# PCA 적용 후 특성 개수 확인
print('PCA 적용 전 특성 개수:', X_train.shape[1])
print('PCA 적용 후 특성 개수:', X_train_pca.shape[1])

display(X_train_pca[:5]) # 생성된 2개변수 5행 출력

# 각 주성분의 설명 분산 비율 확인
print('주성분별 설명 분산 비율:', pca.explained_variance_ratio_)
# [0.44413492 0.18944618] 전체30개변수에 대해 첫번째 변수가 44% 정도설명, 두번째 변수가 19%정도 설명 

# 두 주성분이 보존하는 전체 분산 비율, 두변수 합해서 63%정도 설명
print('누적 설명 분산 비율:', pca.explained_variance_ratio_.sum())


PCA 적용 전 특성 개수: 30
PCA 적용 후 특성 개수: 2


array([[-4.16725097,  0.25551153],
       [ 4.59726035, -0.96763448],
       [-4.56432721, -0.17149634],
       [-0.88369256,  0.43891174],
       [-2.95139888,  0.36563266]])

주성분별 설명 분산 비율: [0.44413492 0.18944618]
누적 설명 분산 비율: 0.6335811006772492


In [7]:
# ==================================================
# 5. PCA로 축소한 2개 특성으로 분류 모델 학습
# ==================================================

# 원본 특성 모델과 동일한 설정 사용
model2 = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

# PCA로 변환된 2개 주성분으로 학습
model2.fit(X_train_pca, y_train)

# 테스트 데이터의 클래스 예측
pred2 = model2.predict(X_test_pca)

In [8]:
# ==================================================
# 6. 두 모델의 성능 비교
# ==================================================

# 정확도와 F1 점수 계산
result = pd.DataFrame({
    'Model': [
        'Original 30 Features',
        'PCA 2 Components'
    ],
    'Accuracy': [
        accuracy_score(y_test, pred1),
        accuracy_score(y_test, pred2)
    ],
    'F1 Score': [
        f1_score(y_test, pred1),
        f1_score(y_test, pred2)
    ]
})

display(result)

,Model,Accuracy,F1 Score
0,Original 30 Features,0.956140,0.965517
1,PCA 2 Components,0.921053,0.937063
